# Deterministic forecast evaluation

This notebook shows how to use `ForecastPerformance` to evaluate **deterministic forecasts** with a suite of skill metrics.  



In [4]:
import pandas as pd
from pathlib import Path
from performance import ForecastPerformance, Results 

test_dataset_path = Path(r'tests\test_datasets_hourly')

obs = pd.read_csv(test_dataset_path / 'observed.csv', sep=';', index_col=0, parse_dates=True)
obs.columns = pd.Index(pd.to_timedelta(obs.columns), name='leadtime')
obs.index.name = 'event_datetime'

det_path = test_dataset_path / 'det.parquet'
det = pd.read_parquet(det_path)
display(det.head(5))

Q
production_datetime event_datetime      leadtime                 
2024-11-26          2024-11-26 01:00:00 0 days 01:00:00  14.05688
                    2024-11-26 02:00:00 0 days 02:00:00  14.31981
                    2024-11-26 03:00:00 0 days 03:00:00  14.82277
                    2024-11-26 04:00:00 0 days 04:00:00  15.31329
                    2024-11-26 05:00:00 0 days 05:00:00  16.04536

In [5]:
results = Results('Model', 'Metric', 'Leadtime')

fp = ForecastPerformance(obs)
fp.add(det, name='det 1')
fp.add(det + 2, name='det 2')
fp.add(det * 1.2, name='det 3')

for model in fp.names():
    print(f'{model}')
    for metric in [fp.KGEprime, fp.NSE, fp.RMSE, fp.MAE, fp.bias, fp.relative_bias, fp.Pearson]:
        for leadtime in det.index.get_level_values('leadtime').unique():
            results.append(Model=model, Metric=metric.__name__, Leadtime=leadtime,
                        Value=fp.deterministic(metric, model, leadtime=leadtime),
                        )
        
df = results.to_pandas(index=['Metric', 'Model'], columns=['Leadtime'])
display(df.iloc[:, ::6])

det 1
det 2
det 3


Value                                  \
Leadtime            0 days 01:00:00 0 days 07:00:00 0 days 13:00:00   
Metric        Model                                                   
bias          det 1        0.237542        0.692919        0.862242   
              det 2        2.237542        2.692919        2.862242   
              det 3        1.783633        2.323527        2.517872   
kge_prime     det 1        0.919410        0.881581        0.857604   
              det 2        0.657637        0.595628        0.574148   
              det 3        0.750693        0.679996        0.650667   
mae           det 1        0.591081        1.300243        1.584032   
              det 2        2.274259        3.000224        3.227270   
              det 3        1.816374        2.627175        2.907780   
nse           det 1        0.956979        0.855309        0.812466   
              det 2        0.862591        0.725135        0.667266   
              det 3        0.791779        0.612151        0.532067   
pearson       det 1        0.986876        0.952188        0.941316   
              det 2        0.986876        0.952188        0.941316   
              det 3        0.986876        0.952188        0.941316   
relative_bias det 1        0.031702        0.092883        0.116269   
              det 2        0.298621        0.360975        0.385960   
              det 3        0.238043        0.311460        0.339523   
rmse          det 1        1.502089        2.743514        3.101741   
              det 2        2.684481        3.781342        4.131557   
              det 3        3.304570        4.491767        4.899558   

                                                                     \
Leadtime            0 days 19:00:00 1 days 01:00:00 1 days 07:00:00   
Metric        Model                                                   
bias          det 1        0.534702        0.077968       -0.328796   
              det 2        2.534702        2.077968        1.671204   
              det 3        2.118089        1.565142        1.074473   
kge_prime     det 1        0.876601        0.884782        0.861053   
              det 2        0.598451        0.614991        0.605368   
              det 3        0.696187        0.758317        0.803273   
mae           det 1        1.674407        1.723159        1.781654   
              det 2        3.173018        3.054489        2.965655   
              det 3        2.808867        2.586505        2.424145   
nse           det 1        0.782814        0.766507        0.746116   
              det 2        0.662292        0.681628        0.693313   
              det 3        0.551933        0.613604        0.659370   
pearson       det 1        0.913736        0.888148        0.868666   
              det 2        0.913736        0.888148        0.868666   
              det 3        0.913736        0.888148        0.868666   
relative_bias det 1        0.072431        0.010597       -0.044764   
              det 2        0.343352        0.282413        0.227525   
              det 3        0.286917        0.212716        0.146284   
rmse          det 1        3.326021        3.444053        3.592918   
              det 2        4.147436        4.021613        3.948908   
              det 3        4.777273        4.430463        4.161703   

                                                     
Leadtime            1 days 13:00:00 1 days 19:00:00  
Metric        Model                                  
bias          det 1       -0.689775       -1.003224  
              det 2        1.310225        0.996776  
              det 3        0.639960        0.263505  
kge_prime     det 1        0.809795        0.746964  
              det 2        0.555348        0.469889  
              det 3        0.813057        0.784084  
mae           det 1        1.920489        2.051122  
              det 2        2.956895        2.980422  
              det 3       